### In this code:
1. We first visualize the trajectory to be filled with meta-atoms.
2. Then we mark the future centers of the meta-atoms with dots.
3. we calculate the phase shift at these points
4. load the trained model, fill in the dots with cells

In [ ]:
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
import numpy as np
import conf

dB = 8 + 2 # side of square of single cell. So It depends on library. Let's count size + 2 microns
APPERTURE_DIAMETER = 100 # # 100 microns in diameters
N_AMOUNT_SPIRAL = APPERTURE_DIAMETER/dB # this quantity for whole diameter
RESOLUTION_SPIRAL = 1000
OFFSET_SPIRAL = dB/2
STEP = 8  # Step between points
focal_length = 100  # Focal length
wavelength = conf.inc_lcen # Wavelength 3 microns

def calculate_phase_shift(diameter, offset, step, focal_length, wavelength):
    radius = diameter / 2
    
    # Checking that the indentation is correct
    if offset >= radius:
        raise ValueError("The X offset must be less than the radius of the circle.")
    
    # Generating angles for the spiral
    theta = np.linspace(0, N_AMOUNT_SPIRAL * np.pi, RESOLUTION_SPIRAL)  # Increase the number of turns and points if necessary.
    
    # Generating radii for the spiral, from (radius - offset) to offset
    r = np.linspace(radius - offset, offset, len(theta))
    
    # Converting from polar coordinates to Cartesian coordinates
    x = r * np.cos(theta)
    y = r * np.sin(theta)

    # We select points with a given step along the spiral
    distances = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
    cumulative_distances = np.insert(np.cumsum(distances), 0, 0)
    # Total length of the path
    total_length = cumulative_distances[-1]
    
    # Uniform partitioning of the path length with a given step
    sampled_distances = np.arange(0, total_length, step)
    
    # Interpolation of x and y coordinates along the path length
    interp_x = interp1d(cumulative_distances, x, kind='linear')
    interp_y = interp1d(cumulative_distances, y, kind='linear')

    # We get the coordinates of the points
    sampled_points = np.column_stack((interp_x(sampled_distances), interp_y(sampled_distances)))

    # Calculation of the phase shift
    phase_shifts = []
    for point in sampled_points:
        d = np.linalg.norm(point)  # the distance to the focus (in this case (0, 0))
        delta_phi = (2 * np.pi / wavelength) * (d - focal_length) % (2 * np.pi) # Normalization to the range [0, 2π]
        normalized_phi = delta_phi - (2 * np.pi) if delta_phi > 0 else delta_phi # Forced normalization to the range [-2π, 0]
        phase_shifts.append(normalized_phi)
    
    # Visualization
    fig, ax = plt.subplots()
    ax.plot(x, y, label='Спираль')
    
    # Drawing a circle
    circle = plt.Circle((0, 0), radius, color='blue', fill=False, linestyle='--', label='Круг')
    ax.add_artist(circle)
    # Adjusting the axes
    ax.set_aspect('equal', adjustable='datalim')
    plt.axhline(0, color='black', linewidth=0.5)
    plt.axvline(0, color='black', linewidth=0.5)
    plt.grid(color='gray', linestyle='--', linewidth=0.5)
    
    plt.figure(figsize=(8, 8))
    plt.plot(x, y, label="spiral")
    plt.scatter(sampled_points[:, 0], sampled_points[:, 1], color='red', label="Dots on the spiral")
    plt.gca().set_aspect('equal', adjustable='box')
    plt.legend()
    plt.grid()
    plt.title("Spiral with marked points")
    plt.show()
    
    return sampled_points, phase_shifts

# Usage example
points, phases = calculate_phase_shift(APPERTURE_DIAMETER, OFFSET_SPIRAL, STEP, focal_length, wavelength)

# Output of results
print("Coordinates of points on the spiral:")
print(points)
print("\nPhase shifts at these points:")
print(phases)


#### The next step is to place the cells at these points
1. We take the value of the phase shift - we pass it through the model - we get the coefficients of the form
2. Fill the cells correctly in a spiral while maintaining the normality to the spiral.
3. Then we'll optimize something...

In [ ]:
type(phases[1])

In [ ]:
#save numpy with data into new file "points" and "phases"
#save in binary format .npy
np.save('points.npy', points)
np.save('phases.npy', phases)
print("The data has been saved successfully!")

In [ ]:
#Let's load a model
import torch

loaded_model = torch.load("model_shape_prediction.pth")
X_new_data = torch.tensor([-3.0])
X_new_data = X_new_data.to(torch.float32) 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
X_new_data = X_new_data.to(device)
loaded_model = loaded_model.to(device)

# Setting the evaluation mode
loaded_model.eval()

# Forecasting
with torch.no_grad():
    prediction = loaded_model(X_new_data)
    
# Output
print("Model output data:", prediction)

In [ ]:
type(prediction)